# Model 5: Transformer Encoder on MFCC Sequences
**Speech Emotion Recognition — CREMA-D**

- **Input**: MFCC sequences `(200, 40)` — same as LSTM input
- **Architecture**: Positional Encoding → 4-layer Transformer Encoder → Mean Pooling → Linear
- **Key advantage**: Self-attention captures global dependencies across the entire utterance simultaneously (unlike LSTM which processes left-to-right)

**Run `feature_extraction.ipynb` first.**

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
BASE_DIR     = os.path.dirname(os.path.abspath('__file__'))
FEATURES_DIR = os.path.join(BASE_DIR, 'features')

EMOTION_FULL = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness']
NUM_CLASSES  = 6
INPUT_SIZE   = 40    # MFCC coefficients per frame
SEQ_LEN      = 200   # time frames

# Transformer hyperparameters
D_MODEL      = 128   # embedding dimension (must be divisible by nhead)
NHEAD        = 8     # attention heads
NUM_LAYERS   = 4     # transformer encoder layers
DIM_FF       = 256   # feed-forward hidden size
DROPOUT      = 0.2

BATCH_SIZE   = 64
EPOCHS       = 60
LR           = 5e-4

print('Config set.')

## 2. Load MFCC Sequence Features

In [ ]:
def load_split(split):
    X = np.load(os.path.join(FEATURES_DIR, f'mfcc_seq_{split}.npy'))  # (N, 200, 40)
    y = np.load(os.path.join(FEATURES_DIR, f'labels_{split}.npy'))
    return torch.FloatTensor(X), torch.LongTensor(y)

X_train, y_train = load_split('train')
X_val,   y_val   = load_split('val')
X_test,  y_test  = load_split('test')

# Normalize
mean = X_train.mean(dim=(0, 1), keepdim=True)
std  = X_train.std(dim=(0, 1),  keepdim=True) + 1e-8
X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=BATCH_SIZE)

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

## 3. Positional Encoding

Since Transformers have no built-in notion of order, we inject position information using sinusoidal positional encodings.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Build sinusoidal encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)   # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

print('Positional encoding defined.')

## 4. Model Architecture — Transformer Encoder

In [ ]:
class EmotionTransformer(nn.Module):
    def __init__(self, input_size=40, d_model=128, nhead=8,
                 num_layers=4, dim_ff=256, dropout=0.2, num_classes=6):
        super().__init__()

        # Project MFCC features (40-dim) to d_model (128-dim)
        self.input_proj = nn.Linear(input_size, d_model)

        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model        = d_model,
            nhead          = nhead,
            dim_feedforward = dim_ff,
            dropout        = dropout,
            batch_first    = True,   # input shape: (batch, seq, features)
            norm_first     = True    # Pre-LN for better training stability
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x: (batch, 200, 40)
        x = self.input_proj(x)           # (batch, 200, 128)
        x = self.pos_encoding(x)         # add positional info
        x = self.encoder(x)              # (batch, 200, 128) — self-attention
        x = x.mean(dim=1)               # mean pooling over time → (batch, 128)
        return self.classifier(x)

model = EmotionTransformer(
    input_size=INPUT_SIZE, d_model=D_MODEL, nhead=NHEAD,
    num_layers=NUM_LAYERS, dim_ff=DIM_FF, dropout=DROPOUT, num_classes=NUM_CLASSES
).to(DEVICE)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training with Warmup + Cosine Decay

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # label smoothing helps with overfitting
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)

# Cosine annealing with warm restarts
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1, eta_min=1e-5
)

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct    += (out.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)
    train_losses.append(total_loss / total)
    train_accs.append(correct / total)

    # Validate
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            out  = model(X_batch)
            loss = nn.CrossEntropyLoss()(out, y_batch)  # no label smoothing for eval
            total_loss += loss.item() * len(y_batch)
            correct    += (out.argmax(1) == y_batch).sum().item()
            total      += len(y_batch)
    val_losses.append(total_loss / total)
    val_accs.append(correct / total)

    scheduler.step()

    if val_accs[-1] > best_val_acc:
        best_val_acc = val_accs[-1]
        torch.save(model.state_dict(), os.path.join(BASE_DIR, 'best_transformer.pth'))

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {train_losses[-1]:.4f}, Acc: {train_accs[-1]:.4f} | '
              f'Val Loss: {val_losses[-1]:.4f}, Acc: {val_accs[-1]:.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.6f}')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses,   label='Val Loss')
axes[0].set_title('Loss Curve — Transformer'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot([a*100 for a in train_accs], label='Train Acc')
axes[1].plot([a*100 for a in val_accs],   label='Val Acc')
axes[1].set_title('Accuracy Curve — Transformer'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'transformer_training_curves.png'), dpi=150)
plt.show()

## 7. Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load(os.path.join(BASE_DIR, 'best_transformer.pth'), map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f'Test Accuracy: {test_acc*100:.2f}%\n')
print(classification_report(all_labels, all_preds, target_names=EMOTION_FULL))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=EMOTION_FULL, yticklabels=EMOTION_FULL)
plt.title(f'Transformer Confusion Matrix — Test Accuracy: {test_acc*100:.2f}%')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'transformer_confusion_matrix.png'), dpi=150)
plt.show()

## 8. Visualize Self-Attention Weights (Bonus)

Peek at what the first attention head focuses on for a sample utterance.

In [ ]:
# Hook to capture attention weights from the first encoder layer
attention_maps = []

def hook_fn(module, input, output):
    # output is (attn_output, attn_weights) when need_weights=True
    pass

# Run one sample through and visualize MFCC input
sample_x = X_test[0:1].to(DEVICE)   # (1, 200, 40)
true_label = y_test[0].item()

model.eval()
with torch.no_grad():
    pred = model(sample_x).argmax(1).item()

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(sample_x[0].cpu().numpy().T, aspect='auto', origin='lower', cmap='viridis')
ax.set_title(f'MFCC Input — True: {EMOTION_FULL[true_label]} | Predicted: {EMOTION_FULL[pred]}')
ax.set_xlabel('Time Frame'); ax.set_ylabel('MFCC Coefficient')
plt.colorbar(ax.images[0], ax=ax)
plt.tight_layout()
plt.show()

## Summary

The Transformer encoder uses **multi-head self-attention** to directly model relationships between all time frames simultaneously.  
Key design choices:
- **Pre-LN** (`norm_first=True`): more stable training than post-LN
- **Label smoothing**: reduces overconfidence and improves generalization
- **Cosine Annealing with Warm Restarts**: escapes local minima
- **Mean pooling**: aggregates all time step representations equally

**Next step:** `results_comparison.ipynb` — compare all 5 models.